In [1]:
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

In [2]:
import pandas as pd
from ITER_DBSCAN import ITER_DBSCAN
from evaluation import EvaluateDataset

2026-03-03 09:18:57.146464: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
filepath = "review_dengan_intent.csv"
df = pd.read_csv(filepath)
df.head(5)

,userName,score,content,at,intent
0,Nabila Livia,5,sukaa,2026-02-18 10:36:04,Praise & Gratitude
1,Antok Sumawan,5,ya bagus,2026-02-18 10:32:49,Praise & Gratitude
2,Khayyira Ira,5,aku suka banget sama tiktok ini,2026-02-18 10:32:45,Praise & Gratitude
3,Aksay Subang,1,akun gua entah kenapa di Ben padalah gua kaga ...,2026-02-18 10:31:33,Account Issue
4,kenzo zildane alvaro,1,tolong diperbaiki,2026-02-18 10:30:44,General Request/Complaint


In [4]:
print('Before: ', len(df))
df = df.dropna()
print('After: ', len(df))
df = df.reset_index()
del df['index']
df.intent.value_counts()

Before:  723
After:  723


intent
Praise & Gratitude           271
Account Issue                129
Performance Issue            100
Feature Complaint/Request     90
Technical Bug/Crash           72
General Request/Complaint     38
Monetization/Earning          23
Name: count, dtype: int64

In [5]:
dataset = df.content.values.tolist()

In [6]:
dataset[0:5]

['sukaa',
 'ya bagus',
 'aku suka banget sama tiktok ini',
 'akun gua entah kenapa di Ben padalah gua kaga ngapa ngapain',
 'tolong diperbaiki']

In [7]:
from IndoSBERTEmbedding import IndoSBERTEmbedding

embedding = IndoSBERTEmbedding()
vectors = embedding.getEmbeddings(dataset)

Loading Huggingface IndoSBERT model....
Model Loaded.


100%|██████████| 723/723 [02:49<00:00,  4.26it/s]


In [8]:
vectors[0]

array([-5.25603473e-01,  3.32280606e-01,  1.15906820e-04,  7.16780126e-03,
       -3.48546714e-01,  5.66146553e-01, -1.64278552e-01,  1.36096060e-01,
        4.64701325e-01, -1.25063986e-01,  1.43200874e-01, -8.04134965e-01,
       -3.73169892e-02,  1.28441285e-02,  6.21265948e-01, -2.59610228e-02,
        4.62310314e-01, -5.78375123e-02, -3.25420499e-01, -4.71290946e-01,
        7.73476779e-01,  2.06139669e-01, -9.32600796e-02,  3.67514193e-02,
       -7.65659437e-02,  1.33830190e-01,  3.22796732e-01, -4.05390114e-01,
        8.88464227e-02,  6.84044242e-01, -6.21281303e-02, -3.38686168e-01,
        1.48874717e-02, -4.33994323e-01,  3.35726529e-01, -4.00579631e-01,
        5.20795763e-01, -3.83500159e-01,  3.51658240e-02, -7.15711296e-01,
        1.68145254e-01,  7.65009046e-01,  5.60266256e-01,  1.62173226e-01,
       -1.10994987e-01, -3.40746850e-01, -4.10492152e-01, -3.72823536e-01,
        3.89402181e-01, -2.32415468e-01,  2.70803869e-01, -7.34851182e-01,
       -1.72696874e-01, -

In [9]:
%%time
model = ITER_DBSCAN(
    initial_distance=0.3,
    initial_minimum_samples=15,
    delta_distance=0.01, 
    delta_minimum_samples=1, 
    max_iteration=15,
    # features="precomputed", # Comment if fit_predict using direct dataset
    # embedding_model="IndoSBERT",  # Uncomment if fit_predict using direct dataset
    # metric="euclidean" # Uncomment if fit_predict using direct dataset
)

CPU times: user 80 μs, sys: 509 μs, total: 589 μs
Wall time: 357 μs


In [10]:
%%time
labels = model.fit_predict(vectors)

CPU times: user 284 ms, sys: 180 ms, total: 464 ms
Wall time: 470 ms


In [11]:
df['cluster_ids'] = labels
df.cluster_ids.value_counts()

cluster_ids
 3     231
 0     131
-1      99
 2      25
 4      18
 1      17
 7      17
 5      10
 14     10
 19     10
 18      8
 8       8
 6       8
 37      8
 9       8
 10      7
 20      7
 27      6
 11      6
 12      6
 13      6
 30      6
 15      6
 17      6
 22      5
 16      5
 21      5
 35      4
 23      4
 28      4
 36      4
 26      4
 24      4
 25      4
 33      4
 34      3
 29      3
 31      3
 32      3
Name: count, dtype: int64

In [12]:
df.to_excel("result-indosbert.xlsx", index=False)

In [13]:
evaluate_dataset = EvaluateDataset(filename=filepath, 
                                   filetype='csv', 
                                   text_column='content', 
                                   target_column='intent')

In [14]:
parameters = [
{
    "distance": 0.3,
    "minimum_samples":30, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "threshold": 1000,
    "embedding_model": "IndoSBERT",
    # "metric": "euclidean"
},
# {
#     "distance": 0.3,
#     "minimum_samples":15, 
#     "delta_distance":0.01, 
#     "delta_minimum_samples":1, 
#     "max_iteration":15,
#     "embedding_model": "IndoSBERT",
#     "threshold": 1000,
#     # "metric": "euclidean"
# },
# {
#     "distance": 0.3,
#     "minimum_samples":15, 
#     "delta_distance":0.01, 
#     "delta_minimum_samples":1, 
#     "max_iteration":15,
#     "embedding_model": "IndoBERT",
#     "threshold": 1000,
#     # "metric": "euclidean"
# },
# {
#     "distance": 0.25,
#     "minimum_samples":15, 
#     "delta_distance":0.01, 
#     "delta_minimum_samples":1, 
#     "max_iteration":15,
#     "embedding_model": "IndoBERT",
#     # "metric": "euclidean"
# },
]

In [15]:
%%time
results = evaluate_dataset.evaulate_iter_dbscan(parameters)
result_df = pd.DataFrame.from_dict(results)

  0%|          | 0/1 [00:00<?, ?it/s]

Loading Huggingface IndoSBERT model....
Model Loaded.


100%|██████████| 1/1 [02:34<00:00, 154.69s/it]

CPU times: user 8min 22s, sys: 27.6 s, total: 8min 49s
Wall time: 2min 34s


In [16]:
result_df

,distance,minimum_samples,delta_distance,delta_minimum_samples,max_iteration,threshold,embedding_model,time,percentage_labelled,clusters,...,adjusted_mutual_info_score,adjusted_rand_score,silhouette_score,davies_bouldin,calinski_harabasz,accuracy,precision,recall,f1,intents
0,0.3,30,0.01,1,15,1000,IndoSBERT,0.17,70.4,9,...,0.24,0.18,0.17,2.141,144.04,0.467497,35.3,46.7,37.9,2


In [17]:
result_df.to_excel("evaluation-result-indosbert.xlsx", index=False)